In [2]:
import Jaccard as J
from EMM_fixed import EMM
from EMM import EMM as oldEMM
from copy import deepcopy

# for data generation
import numpy as np
import pandas as pd




import statsmodels.api as sm



In [3]:
# datasize = 2000             ## GENERATE DATA (exceptional set) ##
# randomvariables = 20


# standard = 10
# predictor = list(np.random.normal(10,1,datasize))
# errorsd = 10
# result = []

# # we doctor our variables in a non random way
# v1 = [0,1,0,1,0]*int(datasize/5)
# v2 = [0]*int(datasize/5*3) + [1]*int(datasize/5*2)


# noisevars = [list(np.random.binomial(1,0.4,datasize)) for _ in range(randomvariables-2)]

# variables = [v1, v2] +noisevars
# # generate result;
# # result data where first two variables are both 1 is different
# for i in range(datasize):
#     v = standard
#     if variables[0][i] == 1 and variables[1][i] == 1:
#         v-=0.5
#     # elif variables[0][i] == 1 and variables[2][i] == 1:
#     #     result.append(4 * predictor[i]   + np.random.normal(0,errorsd) )
#     result.append((v)* predictor[i]  + np.random.normal(0,errorsd) )


# # create a dataframe with number i as column title with the before generated columns
# df = pd.DataFrame({i:ls for i,ls in enumerate(variables)})

# df['result'] = result
# df['predictor'] = predictor

In [4]:
# target_columns = ['predictor','result']
# Beam = EMM(width=20)
# Beam.set_data(deepcopy(df), target_columns)
# Beam.increase_depth()


In [5]:
# Beam.increase_depth(print_result_end=True)
# Beam.beam.print_q()

In [6]:
# JBeam = J.Jaccard_EMM(width=20)
# JBeam.set_data(deepcopy(df), target_columns)
# JBeam.increase_depth(print_result_end=True)


In [7]:
# JBeam.increase_depth(print_result_end=True)
# JBeam.beam.print_q()

In [8]:
# JBeam.increase_depth(print_result_end=True)
# JBeam.beam.print_q()

In [9]:
# datasize = 2000             ## GENERATE DATA (noisy) ##
# randomvariables = 9


# standard = 10
# predictor = list(np.random.normal(10,1,datasize))
# errorsd = 10
# result = []

# # we doctor our variables in a non random way
# v1 = [0,1,0,1,0]*int(datasize/5)
# v2 = [0]*int(datasize/5*3) + [1]*int(datasize/5*2)


# noisevars = [list(np.random.binomial(1,0.4,datasize)) for _ in range(randomvariables-2)]

# variables = [v1, v2] +noisevars
# # generate result;
# # result data where first two variables are both 1 is different
# for i in range(datasize):
#     if variables[0][i] == 1 and variables[1][i] == 1:
#         result.append((standard+3)* predictor[i]  + np.random.normal(0,errorsd) )
#     elif variables[2][i] == 0:
#         result.append((standard+1)*predictor[i] + np.random.normal(0, errorsd) )
#     # elif variables[0][i] == 1 and variables[2][i] == 1:
#     #     result.append(4 * predictor[i]   + np.random.normal(0,errorsd) )
#     else:
#         result.append((standard)* predictor[i]  + np.random.normal(0,errorsd) )


# # create a dataframe with number i as column title with the before generated columns
# df = pd.DataFrame({i:ls for i,ls in enumerate(variables)})

# df['result'] = result
# df['predictor'] = predictor

In [10]:

def generate(datasize = 2000, randomvariables = 20, mean = 10, errorsd = 0.5, exceptionality= 1, pollutionset=4):
    result = []

    predictor = list(np.random.normal(mean,3,datasize))         # STANDARD DEVIATION IS LOCKED AT 3 HERE

    variables = [list(np.random.binomial(1,0.4,datasize)) for _ in range(randomvariables)]

    # generate result;
    # result data where first two variables are both 1 is different
    for i in range(datasize):
        v = 10
        if variables[2][i] == 0:
            v+=pollutionset
        elif variables[0][i] == 1 and variables[1][i] == 1:
            v-=exceptionality
        # elif variables[0][i] == 1 and variables[2][i] == 1:
        #     result.append(4 * predictor[i]   + np.random.normal(0,errorsd) )
        result.append((v)* predictor[i]  + np.random.normal(0,errorsd) )


    # create a dataframe with number i as column title with the before generated columns
    df = pd.DataFrame({i:ls for i,ls in enumerate(variables)})

    df['result'] = result
    df['predictor'] = predictor
    return df

In [11]:
resultdict = {'w': [], 'errormargin':[], 'pollutionplus':[], 'exceptionalplus':[], 'typebeam':[], 'index':[]}
iterate = 2


target_columns = ['predictor','result']
for w in [10,25,60]:
    for margin in [0.5, 2, 10]: #standard deviation
        for pollution in [4, 8, 20]:
            for exceptional in [1, 3, 5]:
                print(f'NORMAL: This run we test with SD of {margin} and pollution set varying by {pollution} and exceptional set varying by {exceptional}')
                df = generate(datasize=2000, randomvariables=20, mean=10, errorsd=margin, pollutionset=pollution, exceptionality=exceptional)
                Beam = EMM(width=w) # smaller width for readability
                Beam.set_data(deepcopy(df), target_columns)
                Beam.increase_depth(iterations=iterate)
                notfound = True
                resultdict['errormargin'].append(margin)            # append relevant information
                resultdict['pollutionplus'].append(pollution) 
                resultdict['exceptionalplus'].append(exceptional)
                resultdict['typebeam'].append('normal') 
                resultdict['w'].append(w)
                for i, sg in enumerate(Beam.beam.calculate_q()):    # search in q for the subgroup
                    if sg.description.description =={1: 1,0: 1}:
                        resultdict['index'].append( i)
                        notfound=False
                        break
                if notfound==True:
                    resultdict['index'].append(9999)                # append 9999 if we can't find the subgroup
                
                print(f'JACCARD: This run we test with SD of {margin} and pollution set varying by {pollution} and exceptional set varying by {exceptional}')
                JBeam = J.Jaccard_EMM(width=w)
                JBeam.set_data(deepcopy(df), target_columns)
                JBeam.increase_depth(iterations=iterate)
                notfound = True
                resultdict['errormargin'].append(margin)            # append relevant information
                resultdict['pollutionplus'].append(pollution) 
                resultdict['exceptionalplus'].append(exceptional)
                resultdict['typebeam'].append('Jaccard') 
                resultdict['w'].append(w)
                for i, sg in enumerate(JBeam.beam.calculate_q()):   # search in q for the subgroup
                    if sg.description.description =={1: 1,0: 1}:
                        resultdict['index'].append(i)
                        notfound=False
                        break
                if notfound==True:
                    resultdict['index'].append(9999)                # append 9999 if we can't find the subgroup
                
                print(f'OLD: This run we test with SD of {margin} and pollution set varying by {pollution} and exceptional set varying by {exceptional}')
                oldBeam = oldEMM(width=w, depth=iterate, evaluation_metric='regression')
                oldBeam.search(deepcopy(df),  target_columns)
                notfound = True
                resultdict['errormargin'].append(margin)            # append relevant information
                resultdict['pollutionplus'].append(pollution) 
                resultdict['exceptionalplus'].append(exceptional)
                resultdict['typebeam'].append('Old') 
                resultdict['w'].append(w)
                for i, sg in enumerate(oldBeam.beam.subgroups):     # oldEMM was implemented poorly, so always stores q in subgroups
                    if sg.description.description =={1: 1,0: 1}:
                        resultdict['index'].append(i)
                        notfound=False
                        break
                if notfound==True:
                    resultdict['index'].append(9999)                # append 9999 if we can't find the subgroup


NORMAL: This run we test with SD of 0.5 and pollution set varying by 4 and exceptional set varying by 1


2025-06-26 13:05:41,819 - INFO - Start
2025-06-26 13:05:41,819 - INFO - Memory usage before downsizing 187.62 MB
2025-06-26 13:05:41,833 - INFO - Memory usage after downsizing 54.81 MB
2025-06-26 13:05:42,189 - INFO - finished an iteration
2025-06-26 13:05:45,463 - INFO - finished an iteration
2025-06-26 13:05:45,463 - INFO - Start
2025-06-26 13:05:45,471 - INFO - Memory usage before downsizing 187.62 MB
2025-06-26 13:05:45,481 - INFO - Memory usage after downsizing 54.81 MB


JACCARD: This run we test with SD of 0.5 and pollution set varying by 4 and exceptional set varying by 1


2025-06-26 13:05:46,036 - INFO - finished an iteration
2025-06-26 13:05:49,771 - INFO - finished an iteration
2025-06-26 13:05:49,771 - INFO - Start
2025-06-26 13:05:49,773 - INFO - Memory usage before downsizing 187.62 MB
2025-06-26 13:05:49,782 - INFO - Memory usage after downsizing 54.81 MB


OLD: This run we test with SD of 0.5 and pollution set varying by 4 and exceptional set varying by 1


OSError: [WinError 1455] The paging file is too small for this operation to complete

In [ ]:
df = pd.DataFrame(resultdict)
df#.to_csv('FirstExperimentWith_w.csv', index=False)


In [ ]:
# from EMM import EMM as oldEMM
# resultdict = {'errormargin':[], 'pollutionplus':[], 'exceptionalplus':[], 'typebeam':[], 'index':[]}

# target_columns = ['predictor','result']
# for margin in [0.5, 2, 10]: #standard deviation
#     for pollution in [4, 8, 20]:
#         for exceptional in [1, 3, 5]:
#             print(f'NORMAL: This run we test with SD of {margin} and pollution set varying by {pollution} and exceptional set varying by {exceptional}')
#             df = generate(datasize=2000, randomvariables=20, mean=10, errorsd=margin, pollutionset=pollution, exceptionality=exceptional)
#             Beam = EMM(width=10) # smaller width for readability
#             Beam.set_data(deepcopy(df), target_columns)
#             Beam.increase_depth(iterations=2)
#             notfound = True
#             resultdict['errormargin'].append(margin)
#             resultdict['pollutionplus'].append(pollution) 
#             resultdict['exceptionalplus'].append(exceptional)
#             resultdict['typebeam'].append('normal') 
#             for i, sg in enumerate(Beam.beam.calculate_q()):
#                 if sg.description.description =={1: 1,0: 1}:
#                     resultdict['index'].append( i)
#                     notfound=False
#                     break
#             if notfound==True:
#                 resultdict['index'].append(9999)
#             print(f'OLD: This run we test with SD of {margin} and pollution set varying by {pollution} and exceptional set varying by {exceptional}')
#             oldBeam = oldEMM(width=10, depth=2, evaluation_metric='regression')
#             oldBeam.search(deepcopy(df),  target_columns)
#             notfound = True
#             resultdict['errormargin'].append(margin)
#             resultdict['pollutionplus'].append(pollution) 
#             resultdict['exceptionalplus'].append(exceptional)
#             resultdict['typebeam'].append('Old') 
#             for i, sg in enumerate(oldBeam.beam.subgroups):
#                 if sg.description.description =={1: 1,0: 1}:
#                     resultdict['index'].append(i)
#                     notfound=False
#                     break
#             if notfound==True:
#                 resultdict['index'].append(9999)


In [ ]:
df

,errormargin,pollutionplus,exceptionalplus,typebeam,index
0,0.5,4,1,normal,9999
1,0.5,4,1,Jaccard,9999
2,0.5,4,1,Old,9999
3,0.5,4,3,normal,2
4,0.5,4,3,Jaccard,2
...,...,...,...,...,...
76,10.0,20,3,Jaccard,3
77,10.0,20,3,Old,9999
78,10.0,20,5,normal,9999
79,10.0,20,5,Jaccard,9999


In [4]:

def generatemoresets(datasize = 2000, randomvariables = 20, mean = 10, errorsd = 0.5, exceptionality= 1, pollutionset=4, superexceptional=2 ):
    result = []
    predictor = list(np.random.normal(mean,3,datasize))         # STANDARD DEVIATION IS LOCKED AT 3 HERE
    variables = [list(np.random.binomial(1,0.4,datasize)) for _ in range(randomvariables)]
    for i in range(5):
        variables.append(list(np.random.binomial(1,0.1,datasize))) #generate 5 sets die specifiek veel pollution kunnen veroorzaken

    # generate result;
    # result data where first two variables are both 1 is different
    for i in range(datasize):
        v = 10
        if variables[2][i] == 0:
            v+=pollutionset
        if variables[3][i] == 0:
            v-=pollutionset
        if variables[0][i] == 1 and variables[1][i] == 1:
            v-=exceptionality
        if variables[4][i] == 1 and variables[5][i] == 1:
            v+=exceptionality
        if variables[3][i] == 1 and variables[7][i] == 1 and variables[8][i] == 1: #reverse of second pollution set
            v+=superexceptional
        if variables[2][i] == 0 and variables[6][i] == 1 and variables[9][i] == 1: #include first pollution set
            v-=superexceptional
        result.append((v)* predictor[i]  + np.random.normal(0,errorsd) )


    # create a dataframe with number i as column title with the before generated columns
    df = pd.DataFrame({i:ls for i,ls in enumerate(variables)})

    df['result'] = result
    df['predictor'] = predictor
    return df

In [ ]:
# df = generatemoresets(superexceptional=8)
# JBeam = J.Jaccard_EMM(width=40)
# JBeam.set_data(deepcopy(df), target_columns)
# JBeam.increase_depth()
# JBeam.beam.print()


2025-06-26 11:53:31,791 - INFO - Start
2025-06-26 11:53:31,793 - INFO - Memory usage before downsizing 226.69 MB
2025-06-26 11:53:31,804 - INFO - Memory usage after downsizing 64.58 MB
2025-06-26 11:53:33,454 - INFO - finished an iteration
2025-06-26 11:53:33,454 - DEBUG - --------------------
2025-06-26 11:53:33,462 - DEBUG - 3 = 1 1.4523802413724984 (804), jaccard: 0.3375
2025-06-26 11:53:33,462 - DEBUG - 3 = 0 0.7754779062328844 (1196), jaccard: 0.4548714883442917
2025-06-26 11:53:33,462 - DEBUG - 2 = 1 0.7263919888325499 (813), jaccard: 0.34015852047556144
2025-06-26 11:53:33,462 - DEBUG - 9 = 1 0.5363192332777369 (762), jaccard: 0.3204272363150868
2025-06-26 11:53:33,462 - DEBUG - 6 = 1 0.5184319362228873 (785), jaccard: 0.3322237017310253
2025-06-26 11:53:33,462 - DEBUG - 2 = 0 0.4057653465760235 (1187), jaccard: 0.44377630787733013
2025-06-26 11:53:33,462 - DEBUG - 7 = 1 0.38643636044094065 (787), jaccard: 0.3304521276595745
2025-06-26 11:53:33,462 - DEBUG - 8 = 1 0.286269485546

In [ ]:
# JBeam.increase_depth()
# JBeam.beam.print()

2025-06-26 11:54:05,444 - INFO - finished an iteration
2025-06-26 11:54:05,444 - DEBUG - --------------------
2025-06-26 11:54:05,444 - DEBUG - 9 = 1 AND 6 = 1 0.6469118753595932 (298), jaccard: 0.33030852994555354
2025-06-26 11:54:05,444 - DEBUG - 3 = 1 AND 8 = 1 0.59059087928351 (320), jaccard: 0.3333333333333333
2025-06-26 11:54:05,444 - DEBUG - 3 = 1 AND 7 = 1 0.5575279546636077 (319), jaccard: 0.3333333333333333
2025-06-26 11:54:05,444 - DEBUG - 3 = 0 AND 2 = 1 0.4927816158897096 (480), jaccard: 0.4240231548480463
2025-06-26 11:54:05,444 - DEBUG - 3 = 0 AND 9 = 1 0.4816923458535067 (435), jaccard: 0.33030852994555354
2025-06-26 11:54:05,444 - DEBUG - 3 = 0 AND 6 = 1 0.3870831098077113 (481), jaccard: 0.43134328358208956
2025-06-26 11:54:05,444 - DEBUG - 3 = 1 AND 2 = 0 0.3749152502155666 (471), jaccard: 0.45577211394302847
2025-06-26 11:54:05,444 - DEBUG - 2 = 0 AND 9 = 0 0.3058448559407116 (727), jaccard: 0.44321608040201005
2025-06-26 11:54:05,444 - DEBUG - 2 = 0 AND 6 = 0 0.298

In [ ]:
# JBeam.increase_depth()
# JBeam.beam.print()

2025-06-26 11:54:25,448 - INFO - finished an iteration
2025-06-26 11:54:25,448 - DEBUG - --------------------
2025-06-26 11:54:25,448 - DEBUG - 3 = 1 AND 8 = 1 AND 7 = 1 0.744973809300514 (131), jaccard: 0.3648068669527897
2025-06-26 11:54:25,448 - DEBUG - 9 = 1 AND 6 = 1 AND 3 = 0 0.32005554885124665 (182), jaccard: 0.47244094488188976
2025-06-26 11:54:25,448 - DEBUG - 3 = 1 AND 7 = 1 AND 2 = 0 0.22859216450554504 (187), jaccard: 0.4329501915708812
2025-06-26 11:54:25,456 - DEBUG - 9 = 1 AND 6 = 1 AND 2 = 0 0.22329346142058484 (192), jaccard: 0.47244094488188976
2025-06-26 11:54:25,456 - DEBUG - 3 = 1 AND 8 = 1 AND 2 = 0 0.2058391739376423 (197), jaccard: 0.48872180451127817
2025-06-26 11:54:25,458 - DEBUG - 3 = 1 AND 8 = 1 AND 9 = 0 0.16398006489730854 (192), jaccard: 0.4699248120300752
2025-06-26 11:54:25,458 - DEBUG - 3 = 1 AND 2 = 0 AND 9 = 0 0.1253913295664401 (282), jaccard: 0.41353383458646614
2025-06-26 11:54:25,458 - DEBUG - 3 = 1 AND 7 = 1 AND 9 = 0 0.11128591464461864 (199)

In [ ]:
# JBeam.beam.print_q()

2025-06-26 11:54:25,478 - DEBUG - calculating q
2025-06-26 11:54:25,478 - DEBUG - --------------------
2025-06-26 11:54:25,478 - DEBUG - 3 = 1 1.4523802413724984 (804), jaccard: 0.3375
2025-06-26 11:54:25,478 - DEBUG - 3 = 0 0.7754779062328844 (1196), jaccard: 0.4548714883442917
2025-06-26 11:54:25,481 - DEBUG - 3 = 1 AND 8 = 1 AND 7 = 1 0.744973809300514 (131), jaccard: 0.3648068669527897
2025-06-26 11:54:25,481 - DEBUG - 2 = 1 0.7263919888325499 (813), jaccard: 0.34015852047556144
2025-06-26 11:54:25,482 - DEBUG - 9 = 1 AND 6 = 1 0.6469118753595932 (298), jaccard: 0.33030852994555354
2025-06-26 11:54:25,483 - DEBUG - 3 = 1 AND 8 = 1 0.59059087928351 (320), jaccard: 0.3333333333333333
2025-06-26 11:54:25,483 - DEBUG - 3 = 0 AND 2 = 1 0.4927816158897096 (480), jaccard: 0.4240231548480463
2025-06-26 11:54:25,483 - DEBUG - 3 = 1 AND 7 = 1 0.5575279546636077 (319), jaccard: 0.3333333333333333
2025-06-26 11:54:25,484 - DEBUG - 9 = 1 0.5363192332777369 (762), jaccard: 0.3204272363150868
202

In [ ]:
resultdict = {'w': [], 'errormargin':[], 'pollutionplus':[], 'exceptionalplus':[], 'superexceptionalplus':[], 'typebeam':[], 'pollution1':[], 'pollution2':[], 'exceptional1':[], 'exceptional2':[], 'exceptionalexclude':[], 'exceptionaloverlap':[]}
iterate = 3


target_columns = ['predictor','result']
for w in [10,25,60]:
    for margin in [0.5, 2, 10]: #standard deviation
        for pollution in [4, 8, 20]:
            for exceptional in [1, 3, 5]:
                for superexceptional in [4,12, 25]:
                    print(f'NORMAL: This run we test with SD of {margin} and pollution set varying by {pollution} and exceptional set varying by {exceptional}')
                    df = generatemoresets(datasize=2000, randomvariables=20, mean=10, errorsd=margin, pollutionset=pollution, exceptionality=exceptional)
                    Beam = EMM(width=w) # smaller width for readability
                    Beam.set_data(deepcopy(df), target_columns)
                    Beam.increase_depth(iterations=iterate)
                    pollution1, pollution2, exceptional1, exceptional2, exceptionalexclude, exceptionaloverlap = False, False, False, False, False, False
                    resultdict['errormargin'].append(margin)            # append relevant information
                    resultdict['pollutionplus'].append(pollution) 
                    resultdict['exceptionalplus'].append(exceptional)
                    resultdict['superexceptionalplus'].append(superexceptional)
                    resultdict['typebeam'].append('normal') 
                    resultdict['w'].append(w)
                    for i, sg in enumerate(Beam.beam.calculate_q()):    # search in q for the subgroup
                        if sg.description.description =={2: 0} and pollution1 == False:
                            resultdict['pollution1'].append( i)
                            pollution1=True
                        if sg.description.description =={3: 0} and pollution2 == False:
                            resultdict['pollution2'].append( i)
                            pollution2=True
                        if sg.description.description =={0:1, 1:1} and exceptional1 == False:
                            resultdict['exceptional1'].append( i)
                            exceptional1=True
                        if sg.description.description =={4:1, 5:1} and exceptional2 == False:
                            resultdict['exceptional2'].append( i)
                            exceptional2=True
                        if sg.description.description == {3:1, 7:1, 8:1} and exceptionalexclude == False:
                            resultdict['exceptionalexclude'].append( i)
                            exceptionalexclude=True
                        if sg.description.description =={2:0, 6:1, 9:1} and exceptionaloverlap == False:
                            resultdict['exceptionaloverlap'].append( i)
                            exceptionaloverlap=True
                    if pollution1==False:
                        resultdict['pollution1'].append(9999)                # append 9999 if we can't find the subgroup
                    if pollution2==False:
                        resultdict['pollution2'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptional1==False:
                        resultdict['exceptional1'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptional2==False:
                        resultdict['exceptional2'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptionalexclude==False:
                        resultdict['exceptionalexclude'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptionaloverlap==False:
                        resultdict['exceptionaloverlap'].append(9999)                # append 9999 if we can't find the subgroup
                    
                    print(f'JACCARD: This run we test with SD of {margin} and pollution set varying by {pollution} and exceptional set varying by {exceptional}')
                    JBeam = J.Jaccard_EMM(width=w)
                    JBeam.set_data(deepcopy(df), target_columns)
                    JBeam.increase_depth(iterations=iterate)
                    notfound = True
                    resultdict['errormargin'].append(margin)            # append relevant information
                    resultdict['pollutionplus'].append(pollution) 
                    resultdict['exceptionalplus'].append(exceptional)
                    resultdict['superexceptionalplus'].append(superexceptional)
                    resultdict['typebeam'].append('Jaccard') 
                    resultdict['w'].append(w)
                    for i, sg in enumerate(JBeam.beam.calculate_q()):   # search in q for the subgroup
                        if sg.description.description =={2: 0} and pollution1 == False:
                            resultdict['pollution1'].append( i)
                            pollution1=True
                        if sg.description.description =={3: 0} and pollution2 == False:
                            resultdict['pollution2'].append( i)
                            pollution2=True
                        if sg.description.description =={0:1, 1:1} and exceptional1 == False:
                            resultdict['exceptional1'].append( i)
                            exceptional1=True
                        if sg.description.description =={4:1, 5:1} and exceptional2 == False:
                            resultdict['exceptional2'].append( i)
                            exceptional2=True
                        if sg.description.description == {3:1, 7:1, 8:1} and exceptionalexclude == False:
                            resultdict['exceptionalexclude'].append( i)
                            exceptionalexclude=True
                        if sg.description.description =={2:0, 6:1, 9:1} and exceptionaloverlap == False:
                            resultdict['exceptionaloverlap'].append( i)
                            exceptionaloverlap=True
                    if pollution1==False:
                        resultdict['pollution1'].append(9999)                # append 9999 if we can't find the subgroup
                    if pollution2==False:
                        resultdict['pollution2'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptional1==False:
                        resultdict['exceptional1'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptional2==False:
                        resultdict['exceptional2'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptionalexclude==False:
                        resultdict['exceptionalexclude'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptionaloverlap==False:
                        resultdict['exceptionaloverlap'].append(9999)                # append 9999 if we can't find the subgroup
                    
                    print(f'OLD: This run we test with SD of {margin} and pollution set varying by {pollution} and exceptional set varying by {exceptional}')
                    oldBeam = oldEMM(width=w, depth=iterate, evaluation_metric='regression')
                    oldBeam.search(deepcopy(df),  target_columns)
                    notfound = True
                    resultdict['errormargin'].append(margin)            # append relevant information
                    resultdict['pollutionplus'].append(pollution) 
                    resultdict['exceptionalplus'].append(exceptional)
                    resultdict['superexceptionalplus'].append(superexceptional)
                    resultdict['typebeam'].append('Old') 
                    resultdict['w'].append(w)
                    for i, sg in enumerate(oldBeam.beam.subgroups):     # oldEMM was implemented poorly, so always stores q in subgroups
                        if sg.description.description =={2: 0} and pollution1 == False:
                            resultdict['pollution1'].append( i)
                            pollution1=True
                        if sg.description.description =={3: 0} and pollution2 == False:
                            resultdict['pollution2'].append( i)
                            pollution2=True
                        if sg.description.description =={0:1, 1:1} and exceptional1 == False:
                            resultdict['exceptional1'].append( i)
                            exceptional1=True
                        if sg.description.description =={4:1, 5:1} and exceptional2 == False:
                            resultdict['exceptional2'].append( i)
                            exceptional2=True
                        if sg.description.description == {3:1, 7:1, 8:1} and exceptionalexclude == False:
                            resultdict['exceptionalexclude'].append( i)
                            exceptionalexclude=True
                        if sg.description.description =={2:0, 6:1, 9:1} and exceptionaloverlap == False:
                            resultdict['exceptionaloverlap'].append( i)
                            exceptionaloverlap=True
                    if pollution1==False:
                        resultdict['pollution1'].append(9999)                # append 9999 if we can't find the subgroup
                    if pollution2==False:
                        resultdict['pollution2'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptional1==False:
                        resultdict['exceptional1'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptional2==False:
                        resultdict['exceptional2'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptionalexclude==False:
                        resultdict['exceptionalexclude'].append(9999)                # append 9999 if we can't find the subgroup
                    if exceptionaloverlap==False:
                        resultdict['exceptionaloverlap'].append(9999)                # append 9999 if we can't find the subgroup
df = pd.DataFrame(resultdict)
df.to_csv('SecondExcperiment.csv', index=False)

NORMAL: This run we test with SD of 0.5 and pollution set varying by 4 and exceptional set varying by 1


2025-06-26 12:29:03,582 - INFO - Start
2025-06-26 12:29:03,582 - INFO - Memory usage before downsizing 226.69 MB
2025-06-26 12:29:03,605 - INFO - Memory usage after downsizing 64.58 MB
2025-06-26 12:29:04,053 - INFO - finished an iteration
2025-06-26 12:29:08,138 - INFO - finished an iteration
2025-06-26 12:29:11,981 - INFO - finished an iteration
2025-06-26 12:29:11,981 - INFO - Start
2025-06-26 12:29:11,981 - INFO - Memory usage before downsizing 226.69 MB
2025-06-26 12:29:12,000 - INFO - Memory usage after downsizing 64.58 MB


JACCARD: This run we test with SD of 0.5 and pollution set varying by 4 and exceptional set varying by 1


2025-06-26 12:29:12,681 - INFO - finished an iteration
2025-06-26 12:29:18,394 - INFO - finished an iteration
2025-06-26 12:29:23,179 - INFO - finished an iteration
2025-06-26 12:29:23,179 - INFO - Start
2025-06-26 12:29:23,184 - INFO - Memory usage before downsizing 226.69 MB
2025-06-26 12:29:23,198 - INFO - Memory usage after downsizing 64.58 MB


OLD: This run we test with SD of 0.5 and pollution set varying by 4 and exceptional set varying by 1
